# Byrne-style Metrics (Colab Notebook)
This notebook computes:
- **S95 (tas)** over **20S–20N land and ocean**
- **ΔRH_land** (from `hurs`, if available)
- **ΔT_ocean** (tas over ocean)
- Outputs a **merged CSV** and a **PNG table**
- (Optional) Figures: A) bar + Byrne band, B) percentile curve, C) land–ocean paired bars, D) mechanism scatters


In [ ]:
# If you're on Colab, run this cell once per runtime.
!pip -q install xarray intake regionmask gcsfs zarr fsspec dask[complete] tqdm


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import intake

# ---------------- User Config ----------------
PANGEOCAT = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
HIST_START, HIST_END = "1980-01-01", "2000-12-31"
FUT_START,  FUT_END  = "2080-01-01", "2100-12-31"
LAT_MIN_TROP, LAT_MAX_TROP = -20, 20
Q = 95  # percentile for S95
BYRNE_MEAN, BYRNE_SIGMA = 1.21, 0.07

# Model set (trim with SUBSET while testing)
ALLOWED = sorted({
    "ACCESS-CM2","ACCESS-ESM1-5","AWI-CM-1-1-MR","CAMS-CSM1-0","CMCC-CM2-SR5","CMCC-ESM2",
    "CNRM-CM6-1","CNRM-CM6-1-HR","CNRM-ESM2-1","CanESM5","EC-Earth3","EC-Earth3-CC",
    "EC-Earth3-Veg","EC-Earth3-Veg-LR","GFDL-CM4","GFDL-ESM4","HadGEM3-GC31-LL","IITM-ESM",
    "INM-CM4-8","INM-CM5-0","IPSL-CM6A-LR","NorESM2-LM","NorESM2-MM","TaiESM1","UKESM1-0-LL"
})
SUBSET = None  # e.g., 6 for quick test
if isinstance(SUBSET, int):
    ALLOWED = ALLOWED[:SUBSET]

# Paths (defaults save to Drive if mounted)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORKDIR = "/content/drive/MyDrive/climate_work" if IN_COLAB else "./climate_work"
OUTDIR  = f"{WORKDIR}/historical_output"
PLOTDIR = f"{OUTDIR}/figures_byrne"
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(PLOTDIR, exist_ok=True)

# Merge target (set to your existing CSV or None)
EXISTING_CSV = os.path.join(OUTDIR, "cmip6_model_metrics_1990_2014_p99_centeredRMSE.csv")  # or None
OUT_CSV = os.path.join(OUTDIR, "metrics_with_byrne_fields.csv")
OUT_PNG = os.path.join(OUTDIR, "metrics_with_byrne_fields.png")

MAKE_PLOTS = True  # set False to skip figures


In [ ]:
def normalize_latlon(obj):
    if "latitude" in obj.coords: obj = obj.rename({"latitude":"lat"})
    if "longitude" in obj.coords: obj = obj.rename({"longitude":"lon"})
    if "lon" in obj.coords and float(obj["lon"].max()) > 180:
        obj = obj.assign_coords(lon=((obj["lon"] + 180) % 360) - 180)
    if "lon" in obj.coords:
        obj = obj.sortby("lon")
    if "lat" in obj.coords and obj.lat.size > 1 and float(obj.lat[0]) > float(obj.lat[-1]):
        obj = obj.sortby("lat")
    return obj

def area_weights(lat):
    latr = np.deg2rad(lat)
    w = np.cos(latr)
    return xr.DataArray(w, coords={"lat": lat}, dims=("lat",))

def get_land_ocean_masks(lat, lon):
    import regionmask
    land_polys = regionmask.defined_regions.natural_earth_v5_0_0.land_110
    mask = land_polys.mask(lon, lat)
    land = mask.notnull()
    band = (lat >= LAT_MIN_TROP) & (lat <= LAT_MAX_TROP)
    land = land.where(band, False).astype(bool)
    ocean = (~land).where(band, False).astype(bool)
    return land, ocean

def open_var_for(model, experiment_id, var_id):
    col = intake.open_esm_datastore(PANGEOCAT)
    cat = col.search(source_id=model, experiment_id=experiment_id, table_id="day", variable_id=var_id)
    if len(cat.df) == 0:
        raise RuntimeError(f"No {experiment_id}/day {var_id} for {model}.")
    try:
        dsets = cat.to_dataset_dict(
            zarr_kwargs={"consolidated": True, "use_cftime": True},
            storage_options={"token": "anon"},
        )
        ds = list(dsets.values())[0]
    except Exception:
        df = cat.df[["member_id","grid_label","zstore"]].drop_duplicates().reset_index(drop=True)
        zurl = df.iloc[0]["zstore"]
        ds = xr.open_zarr(zurl, consolidated=True, storage_options={"token":"anon"}, chunks={})
    ds = normalize_latlon(ds)
    da = ds[var_id]
    if var_id in ("tas", "tasmax"):
        units = da.attrs.get("units","").lower()
        if units in ["k","kelvin"]:
            da = da - 273.15
            da.attrs["units"] = "degree_Celsius"
    return da

def spatial_mean_timeseries(da, w):
    return da.weighted(w).mean(("lat","lon"), skipna=True)

def compute_S95_over_mask(tas_hist, tas_fut, mask_bool):
    w = area_weights(tas_hist["lat"])
    tasH = tas_hist.where(mask_bool)
    tasF = tas_fut.where(mask_bool)

    T_hist = tasH.weighted(w).mean(("lat","lon","time"), skipna=True)
    T_fut  = tasF.weighted(w).mean(("lat","lon","time"), skipna=True)
    dT = float(T_fut - T_hist)

    tsH = spatial_mean_timeseries(tasH, w)
    tsF = spatial_mean_timeseries(tasF, w)

    thr = float(np.nanpercentile(tsH.values, 95))

    meanH_hot = float(tsH.where(tsH >= thr).mean(skipna=True))
    meanF_hot = float(tsF.where(tsF >= thr).mean(skipna=True))
    dT95 = meanF_hot - meanH_hot

    S95 = (dT95 / dT) if np.isfinite(dT) and dT != 0 else np.nan
    return S95, dT, dT95, thr

def compute_delta_RH_land(hurs_hist, hurs_fut, land_mask):
    w = area_weights(hurs_hist["lat"])
    rhH = hurs_hist.where(land_mask)
    rhF = hurs_fut.where(land_mask)
    RH_hist = rhH.weighted(w).mean(("lat","lon","time"), skipna=True)
    RH_fut  = rhF.weighted(w).mean(("lat","lon","time"), skipna=True)
    return float(RH_fut - RH_hist)


In [ ]:
BYRNE_MEAN, BYRNE_SIGMA

def plot_bar_with_byrne(df, savepath):
    models = df["model"].tolist()
    vals = df["S95_land (tas)"].astype(float).values

    plt.figure(figsize=(12,5))
    plt.bar(models, vals)
    plt.axhspan(BYRNE_MEAN-BYRNE_SIGMA, BYRNE_MEAN+BYRNE_SIGMA, alpha=0.2)
    plt.axhline(BYRNE_MEAN, linestyle='--')
    plt.ylabel("S95 (ΔT95/ΔT) — Tropical LAND (tas)")
    plt.xticks(rotation=45, ha="right")
    plt.title("Tropical Land Amplification — S95 (tas) by model")
    plt.tight_layout()
    plt.savefig(savepath, dpi=300)
    plt.close()

def plot_percentile_curve(models_data, savepath):
    percentiles = [50, 75, 90, 95, 97, 99]
    curves = []
    for md in models_data:
        curve = md.get("Sx_curve")
        if curve is None:
            continue
        arr = [curve.get(p, np.nan) for p in percentiles]
        curves.append(arr)
    if not curves:
        return
    arr = np.array(curves, float)
    mean_curve = np.nanmean(arr, axis=0)
    q25 = np.nanpercentile(arr, 25, axis=0)
    q75 = np.nanpercentile(arr, 75, axis=0)

    plt.figure(figsize=(6,5))
    plt.plot(percentiles, mean_curve, marker='o')
    plt.fill_between(percentiles, q25, q75, alpha=0.2)
    plt.axhspan(BYRNE_MEAN-BYRNE_SIGMA, BYRNE_MEAN+BYRNE_SIGMA, alpha=0.1)
    plt.axhline(BYRNE_MEAN, linestyle='--')
    plt.xlabel("Percentile (tas)")
    plt.ylabel("Sx (ΔTx/ΔT) — Tropical LAND")
    plt.title("Multi-model Sx (tas) — Tropical Land")
    plt.tight_layout()
    plt.savefig(savepath, dpi=300)
    plt.close()

def plot_mechanism_scatter(df, savepath_rh, savepath_to):
    sub = df.dropna(subset=["S95_land (tas)", "ΔRH_land (%)"])
    if len(sub) >= 2:
        x = sub["ΔRH_land (%)"].values.astype(float)
        y = sub["S95_land (tas)"].values.astype(float)
        plt.figure(figsize=(6,5))
        plt.scatter(x, y)
        if len(x) >= 2:
            m, b = np.polyfit(x, y, 1)
            xf = np.linspace(np.nanmin(x), np.nanmax(x), 100)
            yf = m*xf + b
            plt.plot(xf, yf)
        plt.xlabel("ΔRH_land (%, 20S–20N)")
        plt.ylabel("S95 (tas), Tropical LAND")
        plt.title("Mechanism check: drier-get-hotter (expect negative slope)")
        plt.tight_layout()
        plt.savefig(savepath_rh, dpi=300)
        plt.close()

    sub2 = df.dropna(subset=["S95_land (tas)", "ΔT_ocean (tas)"])
    if len(sub2) >= 2:
        x2 = sub2["ΔT_ocean (tas)"].values.astype(float)
        y2 = sub2["S95_land (tas)"].values.astype(float)
        plt.figure(figsize=(6,5))
        plt.scatter(x2, y2)
        if len(x2) >= 2:
            m2, b2 = np.polyfit(x2, y2, 1)
            xf2 = np.linspace(np.nanmin(x2), np.nanmax(x2), 100)
            yf2 = m2*xf2 + b2
            plt.plot(xf2, yf2)
        plt.xlabel("ΔT_ocean (°C, 20S–20N)")
        plt.ylabel("S95 (tas), Tropical LAND")
        plt.title("Ocean warming linkage (expect positive slope)")
        plt.tight_layout()
        plt.savefig(savepath_to, dpi=300)
        plt.close()


In [ ]:
def compute_Sx_curve_over_land(tas_hist, tas_fut, land_mask, percentiles=(50,75,90,95,97,99)):
    w = area_weights(tas_hist["lat"])
    tasH = tas_hist.where(land_mask)
    tasF = tas_fut.where(land_mask)
    T_hist = tasH.weighted(w).mean(("lat","lon","time"), skipna=True)
    T_fut  = tasF.weighted(w).mean(("lat","lon","time"), skipna=True)
    dT = float(T_fut - T_hist)
    tsH = tasH.weighted(w).mean(("lat","lon"), skipna=True)
    tsF = tasF.weighted(w).mean(("lat","lon"), skipna=True)
    curve = {}
    for p in percentiles:
        thr = float(np.nanpercentile(tsH.values, p))
        meanH_hot = float(tsH.where(tsH >= thr).mean(skipna=True))
        meanF_hot = float(tsF.where(tsF >= thr).mean(skipna=True))
        dTx = meanF_hot - meanH_hot
        curve[p] = (dTx / dT) if np.isfinite(dT) and dT != 0 else np.nan
    return curve


In [ ]:
rows = []
models_data = []

for model in tqdm(ALLOWED, desc="Byrne metrics"):
    try:
        tas_hist = open_var_for(model, "historical", "tas").sel(time=slice(HIST_START, HIST_END))
        tas_fut  = open_var_for(model, "ssp245",    "tas").sel(time=slice(FUT_START,  FUT_END))

        land_mask, ocean_mask = get_land_ocean_masks(tas_hist["lat"], tas_hist["lon"])

        S95_land, dT_land, dT95_land, thr_land = compute_S95_over_mask(tas_hist, tas_fut, land_mask)
        S95_ocean, dT_ocean, dT95_ocean, thr_ocean = compute_S95_over_mask(tas_hist, tas_fut, ocean_mask)

        # ΔRH over land (optional)
        try:
            hurs_hist = open_var_for(model, "historical", "hurs").sel(time=slice(HIST_START, HIST_END))
            hurs_fut  = open_var_for(model, "ssp245",    "hurs").sel(time=slice(FUT_START,  FUT_END))
            dRH_land = compute_delta_RH_land(hurs_hist, hurs_fut, land_mask)
        except Exception:
            dRH_land = np.nan

        Sx_curve = compute_Sx_curve_over_land(tas_hist, tas_fut, land_mask)

        rows.append({
            "model": model,
            "S95_land (tas)": S95_land,
            "S95_ocean (tas)": S95_ocean,
            "ΔT_land (tas)": dT_land,
            "ΔT_ocean (tas)": dT_ocean,
            "ΔRH_land (%)": dRH_land,
            "thr95_hist_land (tas)": thr_land
        })
        models_data.append({"model": model, "Sx_curve": Sx_curve})

        import gc
        del tas_hist, tas_fut
        gc.collect()

        pd.DataFrame(rows).to_csv(OUT_CSV.replace(".csv","_partial.csv"), index=False)

    except Exception as e:
        print(f"[skip] {model}: {e}")
        continue

add = pd.DataFrame(rows).sort_values("model")
add.head()


In [ ]:
if isinstance(EXISTING_CSV, str) and os.path.exists(EXISTING_CSV):
    base = pd.read_csv(EXISTING_CSV)
    model_col = None
    for c in base.columns:
        if c.strip().lower() in ("model","models","name","source_id"):
            model_col = c
            break
    if model_col is None:
        raise ValueError("Couldn't find a model column in EXISTING_CSV.")
    merged = base.merge(add, left_on=model_col, right_on="model", how="left")
    merged.drop(columns=["model"], inplace=True)
    df = merged
else:
    df = add

df.to_csv(OUT_CSV, index=False)
df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 0.5 + 0.35*len(df)))
ax.axis('off')
tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.2)
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight')
print("Saved CSV:", OUT_CSV)
print("Saved PNG:", OUT_PNG)

if MAKE_PLOTS:
    plot_bar_with_byrne(add, savepath=os.path.join(PLOTDIR, "A_bar_with_byrne.png"))
    plot_percentile_curve(models_data, savepath=os.path.join(PLOTDIR, "B_percentile_curve.png"))

    models = add["model"].tolist()
    land_vals = add["S95_land (tas)"].astype(float).values
    ocean_vals = add["S95_ocean (tas)"].astype(float).values
    x = np.arange(len(models))
    width = 0.35
    plt.figure(figsize=(12,5))
    plt.bar(x - width/2, land_vals, width)
    plt.bar(x + width/2, ocean_vals, width)
    plt.axhspan(BYRNE_MEAN-BYRNE_SIGMA, BYRNE_MEAN+BYRNE_SIGMA, alpha=0.2)
    plt.axhline(BYRNE_MEAN, linestyle='--')
    plt.xticks(x, models, rotation=45, ha='right')
    plt.ylabel("S95 (ΔT95/ΔT)")
    plt.title("Paired S95 (tas): Land vs Ocean (20S–20N)")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTDIR, "C_ocean_land_paired.png"), dpi=300)
    plt.close()

    plot_mechanism_scatter(add,
                           savepath_rh=os.path.join(PLOTDIR, "D_scatter_S95_vs_dRH.png"),
                           savepath_to=os.path.join(PLOTDIR, "D_scatter_S95_vs_dT_ocean.png"))

try:
    mm = pd.to_numeric(add["S95_land (tas)"], errors="coerce").mean()
    print(f"Multi-model mean S95 (tas, land): {mm:.3f}  (Byrne ~ 1.21 ± 0.07)")
except Exception:
    pass
